**Import Lib**

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
import os

# Tên thư mục dự án của bạn trên Drive
project_path = "/content/drive/MyDrive/FakeNewsDetection_Project"

# Danh sách các thư mục con cần có (giống link Github bạn gửi)
sub_folders = [
    'Dataset',      # Nơi để file True.csv, Fake.csv
    'Models',       # Nơi lưu mô hình sau khi training
    'Notebooks',    # Nơi lưu file code (.ipynb)
    'Preprocessing' # Nơi lưu các file xử lý ngôn ngữ
]

# Lệnh tạo thư mục
if not os.path.exists(project_path):
    os.makedirs(project_path)

for folder in sub_folders:
    full_path = os.path.join(project_path, folder)
    if not os.path.exists(full_path):
        os.makedirs(full_path)
        print(f"Đã tạo thư mục: {full_path}")
    else:
        print(f"Thư mục đã tồn tại: {full_path}")

Đã tạo thư mục: /content/drive/MyDrive/FakeNewsDetection_Project/Dataset
Đã tạo thư mục: /content/drive/MyDrive/FakeNewsDetection_Project/Models
Đã tạo thư mục: /content/drive/MyDrive/FakeNewsDetection_Project/Notebooks
Đã tạo thư mục: /content/drive/MyDrive/FakeNewsDetection_Project/Preprocessing


In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 60.7 MB/s eta 0:00:00


In [12]:
# Cài đặt thư viện còn thiếu
!pip install gensim

import pandas as pd
import numpy as np
import os
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

# Kết nối Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Prepare Data**

In [13]:
# Đường dẫn dự án bạn đã tạo trên Drive
path_project = "/content/drive/MyDrive/FakeNewsDetection_Project"
path_dataset = os.path.join(path_project, "Dataset")

# Kiểm tra file có trong thư mục Dataset chưa
if not os.path.exists(os.path.join(path_dataset, "True.csv")):
    print("LỖI: Bạn chưa kéo file True.csv và Fake.csv vào thư mục Dataset trên Drive!")
else:
    # Đọc dữ liệu
    df_true = pd.read_csv(os.path.join(path_dataset, "True.csv"))
    df_fake = pd.read_csv(os.path.join(path_dataset, "Fake.csv"))

    # Gán nhãn
    df_true['label'] = 1
    df_fake['label'] = 0

    # Gộp dữ liệu
    df = pd.concat([df_true, df_fake], ignore_index=True)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    df = df[['text', 'label']].dropna()

    print(f"Bước 2 thành công! Tổng số mẫu: {len(df)}")

/tmp/ipykernel_4952/968809726.py:11: DtypeWarning: Columns (4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171) have mixed types. Specify dtype option on import or set low_memory=False.
  df_fake = pd.read_csv(os.path.join(path_dataset, "Fake.csv"))


Bước 2 thành công! Tổng số mẫu: 44919


**Prepare Training Data**

In [14]:
# Tiền xử lý: Tách từ
df['tokenized_text'] = df['text'].apply(lambda x: simple_preprocess(str(x)))

# Chia dữ liệu theo tỷ lệ 80/20
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['tokenized_text'],
    df['label'],
    test_size=0.2,
    random_state=42
)
print("Đã chia dữ liệu xong.")

Đã chia dữ liệu xong.


**Word2Vec Model**

In [15]:
# Huấn luyện mô hình Word2Vec
w2v_model = Word2Vec(sentences=X_train_raw, vector_size=100, window=5, min_count=2, workers=4)

# Hàm chuyển bài báo thành vector số
def get_avg_vec(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv.key_to_index]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

X_train = np.array([get_avg_vec(text, w2v_model) for text in X_train_raw])
X_test = np.array([get_avg_vec(text, w2v_model) for text in X_test_raw])
print("Đã chuyển đổi văn bản sang Vector.")

Đã chuyển đổi văn bản sang Vector.


**Training with Naive Bayes**

In [16]:
# Huấn luyện NB
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

# Dự đoán
y_pred = nb_model.predict(X_test)

# Xuất 4 tiêu chí APRF
print("\n=== KẾT QUẢ THỰC NGHIỆM (APRF) ===")
print(f"Accuracy (A):  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision (P): {precision_score(y_test, y_pred):.4f}")
print(f"Recall (R):    {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score (F):  {f1_score(y_test, y_pred):.4f}")


=== KẾT QUẢ THỰC NGHIỆM (APRF) ===
Accuracy (A):  0.9039
Precision (P): 0.8803
Recall (R):    0.9261
F1-Score (F):  0.9026


In [19]:
def save_metadata(model_name, vector_type, metrics):
    metadata_path = os.path.join(path_models, "model_summary.csv")

    # Tạo dữ liệu mới
    new_data = pd.DataFrame([{
        'Model Name': model_name,
        'Embedding': vector_type,
        'Accuracy': metrics['acc'],
        'Precision': metrics['pre'],
        'Recall': metrics['rec'],
        'F1': metrics['f1']
    }])

    # Nếu file chưa tồn tại thì tạo mới, nếu có rồi thì ghi thêm dòng mới (append)
    if not os.path.isfile(metadata_path):
        new_data.to_csv(metadata_path, index=False, encoding="utf-8")
    else:
        new_data.to_csv(metadata_path, mode='a', header=False, index=False, encoding="utf-8")

    print("Đã cập nhật bảng so sánh mô hình!")

# Cách dùng:
my_metrics = {'acc': 0.92, 'pre': 0.91, 'rec': 0.93, 'f1': 0.92}
save_metadata("nb_classifier.pkl", "Word2Vec_100", my_metrics)

Đã cập nhật bảng so sánh mô hình!


**Test Sentence Real or Fake**

In [17]:
def predict_news(sentence):
    tokens = simple_preprocess(sentence)
    vector = get_avg_vec(tokens, w2v_model).reshape(1, -1)
    res = nb_model.predict(vector)[0]
    return "TIN THẬT" if res == 1 else "TIN GIẢ"

# Nhập câu bất kỳ để test
cau_hoi = "The President signed the new law today."
print(f"\nKết quả dự đoán cho câu '{cau_hoi}':")
print(predict_news(cau_hoi))


Kết quả dự đoán cho câu 'The President signed the new law today.':
TIN THẬT


In [18]:
import pickle
import os

# Đường dẫn đến ngăn chứa Models trong dự án của bạn
path_models = "/content/drive/MyDrive/FakeNewsDetection_Project/Models"

# 1. Lưu mô hình Word2Vec (Định dạng riêng của Gensim)
w2v_model.save(os.path.join(path_models, "w2v_word_embedding.model"))

# 2. Lưu mô hình Naive Bayes (Dùng thư viện pickle)
with open(os.path.join(path_models, "nb_classifier.pkl"), 'wb') as f:
    pickle.dump(nb_model, f)

print("--- ĐÃ LƯU TẤT CẢ MODEL VÀO GOOGLE DRIVE ---")

--- ĐÃ LƯU TẤT CẢ MODEL VÀO GOOGLE DRIVE ---
